# Chapter Review - Refactored

This notebook demonstrates the refactored analysis engine for chapter-by-chapter manuscript analysis.

In [ ]:
# Import the modular components
from analysis_engine import ManuscriptAnalyzer, AnalysisConfig
from prompts import prompt_library
from output_manager import OutputManager

In [ ]:
# Configuration
config = AnalysisConfig(
    model="gpt-4-0125-preview",
    api_delay=2.0,
    temperature=0.7
)

# Initialize components
analyzer = ManuscriptAnalyzer(config)
output_manager = OutputManager()

In [ ]:
# Load manuscript
manuscript_path = "/Users/douglashindson/workspace/blog/tabum/1-3-1-department-of-life.md"
text = analyzer.load_manuscript(manuscript_path)

# Get manuscript info
info = analyzer.get_manuscript_info(text)
print(f"Manuscript Info:")
print(f"  Tokens: {info['token_count']:,}")
print(f"  Chapters: {info['chapter_count']}")
print(f"  Est. Pages: {info['estimated_pages']}")

In [ ]:
# Get prompts - you can use all chapter prompts or select specific ones
prompts = prompt_library.chapter_analysis_prompts

# Or select specific prompts:
# prompts = prompt_library.get_custom_prompts([
#     "character_analysis.txt",
#     "dialogue_evaluation.txt",
#     "pacing_analysis.txt"
# ])

print(f"Using {len(prompts)} analysis prompts:")
for name, _ in prompts:
    print(f"  - {name}")

In [ ]:
# Perform chapter analysis
print("Starting chapter analysis...")
results = analyzer.analyze_by_chapters(text, prompts)
print("\nAnalysis complete!")

In [ ]:
# Save results
output_folder = output_manager.save_chapter_analysis(results, manuscript_path)
output_manager.save_results_json(output_folder, results)

print(f"Results saved to: {output_folder}")

# Show summary
total_results = sum(len(chapter_results) for chapter_results in results.values())
error_count = sum(
    sum(1 for r in chapter_results if r.error) 
    for chapter_results in results.values()
)
success_count = total_results - error_count
print(f"Summary: {success_count} successful, {error_count} errors across {len(results)} chapters")

In [ ]:
# Optional: Display first result as example
if results:
    first_chapter = list(results.keys())[0]
    first_result = results[first_chapter][0]
    
    print(f"\nExample result from Chapter {first_chapter}:")
    print(f"Prompt: {first_result.prompt_name}")
    print(f"Bad Cop Response (first 200 chars): {first_result.bad_cop_response[:200]}...")